# Séance 2 — Prise en main de l'environnement et cas d'usage de l'IA
**Urbanisation des SI à l'ère de l'IA** · Master 2 CPI · Université Paris 8<br>
**Enseignant :** Pr Mehdi Ammi — Laboratoire LIASD (EA 4383)<br>
**Version :** octobre 2026

*Note sur l'usage de l'IA : la progression pédagogique et le scénario ont été définis par l'auteur. Claude (Anthropic) a été utilisé pour la rédaction des exercices et la mise en forme des supports.*

---

> **🎯 Objectifs de la séance**
> - Installer l'environnement du cours et appeler un modèle de langage (local ou API)
> - Explorer le SI simulé de Saint-Denis : applications, données, doublons issus de la fusion
> - Identifier et prioriser les cas d'usage de l'IA pour le guichet citoyen
> - Placer les premières briques IA sur la cartographie : la cible v0

> **Organisation des travaux pratiques**
> - Le travail se fait **en groupe** (4 ou 5 étudiants), sur le fil rouge de la Ville de Saint-Denis.
> - Les cellules **▶️ DÉMONSTRATION** sont fournies : exécutez-les (`Maj + Entrée`) et lisez le résultat. Vous n'avez pas à les modifier.
> - Les cellules **✏️ À VOUS** demandent de modifier ce qui est signalé par `...` ou `TODO` : un texte, une valeur, une liste. Vous n'avez jamais de programme complet à écrire.
> - Les blocs *Vos réponses* se remplissent directement dans le notebook (double-clic sur la cellule).
> - Le SI de Saint-Denis utilisé ici est **fictif** : applications, données et documents ont été inventés pour le cours.

---
## 📩 La commande du jour

> **De :** la Directrice générale des services<br>
> **À :** l'équipe projet « Guichet citoyen augmenté »<br>
> **Objet :** Par où commencer ?
>
> Bonjour à toutes et à tous,
>
> Merci pour votre note de diagnostic. L'élue au numérique souhaite des résultats visibles rapidement,
> mais je ne veux pas d'un projet IA de plus qui s'ajoute à un SI déjà compliqué depuis la fusion avec Pierrefitte.
>
> Je vous demande, pour la fin de la séance :
> 1. un état des lieux chiffré de nos applications (doublons, coûts, satisfaction) ;
> 2. une liste priorisée des démarches où l'IA apporterait le plus, avec les risques ;
> 3. un premier schéma montrant où ces briques IA s'insèrent dans notre SI.
>
> Bien cordialement,
> La DGS

---
# Partie 1 — Prise en main de l'environnement
---

### Rappels

#### Un notebook, comment ça marche ?

Un notebook est une suite de **cellules** :
- les cellules de **texte** (comme celle-ci) donnent les explications ;
- les cellules de **code** s'exécutent avec `Maj + Entrée`. Le résultat s'affiche juste en dessous.

Les cellules s'exécutent **dans l'ordre**. Si une cellule affiche une erreur `NameError`, c'est souvent qu'une cellule précédente n'a pas été exécutée. En cas de doute : menu *Kernel → Restart Kernel and Run All Cells*.

#### Le modèle de langage : local ou API

Tout le cours passe par un seul fichier, `common/llm.py`. Le choix du modèle se fait **uniquement dans le fichier `.env`** (voir le `README.md` de la séance) :

| Option | Où tourne le modèle ? | Avantages | Limites |
| --- | --- | --- | --- |
| **A — Local (Ollama)** | Sur votre ordinateur | Gratuit, aucune donnée ne sort | Plus lent, moins performant, demande de la mémoire |
| **B — API** | Chez un fournisseur | Rapide, plus performant | Payant à l'usage, les données partent chez le fournisseur |

Ce choix n'est pas seulement technique : c'est une **décision d'urbanisation** (souveraineté, coût, dépendance à un fournisseur) que nous retrouverons en séance 7.

La cellule suivante vérifie l'installation. Elle doit se terminer par **« OK : l'environnement est prêt »**.

In [ ]:
!python common/check_setup.py

In [ ]:
# ▶️ DÉMONSTRATION — exécutez et observez
from common.llm import get_llm, description_modele

llm = get_llm()
print("Modèle utilisé :", description_modele())

reponse = llm.invoke("En une phrase, qu'est-ce qu'un système d'information dans une mairie ?")
print(reponse.content)

Un appel au modèle envoie en réalité une **liste de messages** : un message `system` (les consignes, écrites par le concepteur du service) et un message `user` (la demande de l'habitant).

In [ ]:
# ▶️ DÉMONSTRATION — exécutez et observez
messages = [
    ("system", "Tu es un agent d'accueil de la mairie de Saint-Denis. Tu réponds poliment, en deux phrases maximum."),
    ("user", "Bonjour, je viens d'emménager, je dois faire quoi pour la cantine de mon fils ?"),
]
print(llm.invoke(messages).content)

## Exercice 1 — Le rôle des consignes

> **Modalité :** modifier le texte des consignes et observer.

Modifiez la consigne `system` pour obtenir successivement :
1. un agent qui répond **en anglais**, quelle que soit la langue de la question ;
2. un agent qui **refuse** de répondre aux questions qui ne concernent pas la mairie ;
3. un agent qui termine toujours par : « Pour toute démarche, rendez-vous sur le portail famille. »

Testez chaque version avec la question ci-dessous, puis avec une question hors sujet (« Quel temps fera-t-il demain ? »).

In [ ]:
# ✏️ À VOUS
consigne = "..."   # TODO : votre consigne
question = "Bonjour, je viens d'emménager, je dois faire quoi pour la cantine de mon fils ?"

print(llm.invoke([("system", consigne), ("user", question)]).content)

*Vos réponses : la consigne est-elle toujours respectée ? Que se passe-t-il avec la question hors sujet ?*

- …

---
# Partie 2 — Explorer le SI de Saint-Denis
---

### Rappels

#### Le SI simulé

Le module `common/si.py` joue le rôle des applications de la ville. Il contient :
- l'**inventaire** des 20 applications du SI ;
- des **API** : consulter le dossier d'une famille, inscrire un enfant à la cantine, demander une carte de stationnement, etc. ;
- un **journal** qui garde la trace de chaque appel.

#### Le contexte de la fusion

Depuis le 1er janvier 2025, Saint-Denis et Pierrefitte forment une commune nouvelle. Les deux SI ont été réunis, mais pas encore **urbanisés** : plusieurs applications font la même chose. Par ailleurs, certaines compétences (voirie, propreté) relèvent de **Plaine Commune**, qui a son propre SI.

In [ ]:
# ▶️ DÉMONSTRATION — exécutez et observez
from common import si

inventaire = si.inventaire_applications()
inventaire.head(10)

In [ ]:
# ▶️ DÉMONSTRATION — exécutez et observez
# Combien d'applications par origine ?
inventaire["origine"].value_counts()

In [ ]:
# ▶️ DÉMONSTRATION — exécutez et observez
# Les fonctions assurées par plusieurs applications : les doublons de la fusion
par_fonction = inventaire.groupby("fonction")["nom"].apply(list)
par_fonction[par_fonction.apply(len) > 1]

In [ ]:
# ▶️ DÉMONSTRATION — exécutez et observez
import matplotlib.pyplot as plt

cout_par_domaine = inventaire.groupby("domaine")["cout_annuel_keur"].sum().sort_values()
cout_par_domaine.plot.barh(figsize=(8, 4), color="#5b3fd1")
plt.xlabel("Coût annuel (k€)")
plt.title("Coût annuel des applications par domaine")
plt.show()

## Exercice 2 — L'état des lieux chiffré

> **Modalité :** compléter les filtres signalés par `...`.

Un filtre s'écrit `inventaire[inventaire["colonne"] == "valeur"]`. Répondez aux questions de la DGS :

In [ ]:
# ✏️ À VOUS
# a) Les applications au statut "doublon fusion"
doublons = inventaire[inventaire["statut"] == "..."]   # TODO
print("Nombre de doublons :", len(doublons))
print("Coût annuel des doublons (k€) :", doublons["cout_annuel_keur"].sum())

# b) Les applications dont la satisfaction est inférieure ou égale à 2
peu_satisfaisantes = inventaire[inventaire["satisfaction"] <= ...]   # TODO
print(peu_satisfaisantes[["nom", "origine", "satisfaction"]])

# c) Les applications mises en service avant 2012
anciennes = inventaire[inventaire["annee_mise_en_service"] < ...]   # TODO
print(anciennes[["nom", "annee_mise_en_service", "statut"]])

Le SI expose aussi des **API**. Testez-les : chaque appel est enregistré dans le journal.

In [ ]:
# ▶️ DÉMONSTRATION — exécutez et observez
print(si.cantine_consulter_famille("FAM001"))
print(si.cantine_tarif_repas(620))
si.journal()

## Exercice 3 — L'IA comme assistant de l'urbaniste

> **Modalité :** exécuter, puis critiquer la réponse du modèle.

On donne l'inventaire au modèle et on lui demande un scénario de rationalisation. **Ne prenez pas sa réponse pour argent comptant** : comparez-la à votre propre analyse.

In [ ]:
# ▶️ DÉMONSTRATION — exécutez et observez
demande = f"""Voici l'inventaire des applications d'une commune issue d'une fusion :

{inventaire.to_csv(index=False)}

Propose un scénario de rationalisation en 5 points maximum : quelles applications garder,
fusionner ou retirer, et dans quel ordre. Justifie chaque point en une phrase."""

print(llm.invoke(demande).content)

*Vos réponses : le modèle a-t-il vu tous les doublons ? A-t-il commis des erreurs (applications inventées, chiffres faux) ? Qu'a-t-il oublié de prendre en compte (Plaine Commune, les agents, les usagers) ?*

| Ce que l'IA a bien vu | Ses erreurs ou oublis |
| --- | --- |
| | |

---
# Partie 3 — Identifier et prioriser les cas d'usage de l'IA
---

### Rappels

#### Une grille de priorisation

Un cas d'usage est noté de 1 (faible) à 5 (fort) sur quatre critères :

| Critère | Question à se poser |
| --- | --- |
| **Valeur** | Combien d'habitants ou d'agents en bénéficient ? Quel temps gagné ? |
| **Faisabilité** | Les applications concernées ont-elles des API ? La démarche est-elle simple ? |
| **Données** | Les documents et données nécessaires existent-ils, sont-ils à jour ? |
| **Risque** | Quelles conséquences en cas d'erreur ? Données personnelles ? Décision sur une personne ? |

#### Le niveau de risque selon l'AI Act

Le règlement européen sur l'IA classe les systèmes selon leur risque :
- **minimal** : aucune obligation particulière (par exemple un filtre anti-spam) ;
- **limité** : obligation de **transparence** (l'usager doit savoir qu'il parle à une IA) ;
- **haut risque** : obligations fortes (documentation, contrôle humain, qualité des données). C'est notamment le cas des systèmes qui décident de l'**accès à des prestations publiques essentielles** ;
- **inacceptable** : interdit (par exemple la notation sociale des citoyens).

## Exercice 4 — Noter les cas d'usage

> **Modalité :** compléter les notes et ajouter deux cas d'usage de votre choix.

Six cas d'usage ont été proposés lors d'un premier atelier. Notez-les, puis ajoutez **deux cas d'usage** issus de votre diagnostic de la séance 1.

In [ ]:
# ✏️ À VOUS
import pandas as pd

cas_usage = [
    # nom, valeur, faisabilité, données, risque, niveau AI Act
    {"cas": "Répondre aux questions des habitants (horaires, démarches)", "valeur": 5, "faisabilite": 4, "donnees": 4, "risque": 2, "ai_act": "limité"},
    {"cas": "Inscrire un enfant à la cantine de bout en bout",           "valeur": 4, "faisabilite": 3, "donnees": 3, "risque": 3, "ai_act": "..."},
    {"cas": "Demander une carte de stationnement résident",              "valeur": ..., "faisabilite": ..., "donnees": ..., "risque": ..., "ai_act": "..."},
    {"cas": "Transmettre les signalements de voirie à Plaine Commune",    "valeur": ..., "faisabilite": ..., "donnees": ..., "risque": ..., "ai_act": "..."},
    {"cas": "Décider automatiquement des dérogations scolaires",          "valeur": ..., "faisabilite": ..., "donnees": ..., "risque": ..., "ai_act": "..."},
    {"cas": "Résumer les délibérations du conseil municipal",             "valeur": ..., "faisabilite": ..., "donnees": ..., "risque": ..., "ai_act": "..."},
    # TODO : vos deux cas d'usage
    {"cas": "...", "valeur": ..., "faisabilite": ..., "donnees": ..., "risque": ..., "ai_act": "..."},
    {"cas": "...", "valeur": ..., "faisabilite": ..., "donnees": ..., "risque": ..., "ai_act": "..."},
]

grille = pd.DataFrame(cas_usage)
grille["score"] = grille["valeur"] + grille["faisabilite"] + grille["donnees"] - grille["risque"]
grille.sort_values("score", ascending=False)

In [ ]:
# ▶️ DÉMONSTRATION — exécutez et observez
fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(grille["risque"], grille["valeur"], s=grille["faisabilite"] * 80, color="#5b3fd1", alpha=0.6)
for _, ligne in grille.iterrows():
    ax.annotate(ligne["cas"][:35], (ligne["risque"], ligne["valeur"]), fontsize=8,
                xytext=(5, 5), textcoords="offset points")
ax.set_xlabel("Risque →")
ax.set_ylabel("Valeur →")
ax.set_xlim(0.5, 5.5); ax.set_ylim(0.5, 5.5)
ax.axvline(3, color="grey", ls="--", lw=0.8); ax.axhline(3, color="grey", ls="--", lw=0.8)
ax.set_title("Matrice valeur / risque (taille = faisabilité)")
plt.show()

*Vos réponses : quels sont vos trois cas d'usage prioritaires ? Pourquoi écarter certains cas à forte valeur ? Le cas « décider automatiquement des dérogations » vous semble-t-il acceptable ?*

1. …
2. …
3. …

---
# Partie 4 — La cible v0 : placer l'IA dans le SI
---

### Rappels

#### Les quatre couches de la cartographie

| Couche | Ce qu'elle décrit | Exemple à Saint-Denis |
| --- | --- | --- |
| **Métier** | Les processus, les démarches | Inscription à la cantine |
| **Fonctionnelle** | Les services rendus, indépendamment des logiciels | Gestion des dossiers usagers |
| **Applicative** | Les logiciels | Portail famille SD |
| **Données et technique** | Les données, documents et infrastructures | Base des usagers |

Une brique IA n'est pas « à côté » du SI : elle consomme des données, appelle des applications et transforme des processus. La placer sur la cartographie oblige à répondre à ces questions **avant** de la construire.

#### Le module de cartographie

```python
carte.ajouter_brique("Nom de la brique", couche="applicative", seance="S2")
```
Couches possibles : `metier`, `fonctionnelle`, `applicative`, `donnees`, `gouvernance`.

In [ ]:
# ▶️ DÉMONSTRATION — exécutez et observez
from common.carto import Cartographie

carte = Cartographie.charger()      # le SI existant
carte.afficher()

## Exercice 5 — La cible v0

> **Modalité :** ajouter les briques IA de vos cas d'usage prioritaires.

Pour chacun de vos trois cas d'usage prioritaires, ajoutez au moins une brique, dans la bonne couche. Un exemple est donné.

In [ ]:
# ✏️ À VOUS
carte = Cartographie.charger()

carte.ajouter_brique("Démarches confiées à l'IA (priorisées)", couche="metier", seance="S2")
carte.ajouter_brique("Réponse aux questions des habitants", couche="fonctionnelle", seance="S2")
# TODO : vos briques
carte.ajouter_brique("...", couche="...", seance="S2")
carte.ajouter_brique("...", couche="...", seance="S2")

carte.afficher("Saint-Denis — cible v0")
carte.sauvegarder("livrables/cartographie_S2.json")

*🏙️ Question d'urbanisation : pour chaque brique ajoutée, quelles **données** consomme-t-elle et quelles **applications** devra-t-elle appeler ? Lesquelles de ces applications sont des doublons de la fusion ?*

- …

---
# 📦 Livrable 2 — Diagnostic chiffré et cible v0

> **📥 Dépôt sur Moodle — à la fin de la séance**<br>
> Un dépôt par groupe dans l'espace **« Livrable 2 »** du cours : ce notebook exécuté (`S2_environnement_cas_usage.ipynb`) et le fichier `livrables/cartographie_S2.json`.

**Contenu attendu** (dans les cellules *Vos réponses* du notebook) :
1. l'état des lieux chiffré (exercice 2) : nombre et coût des doublons, applications à moderniser ;
2. votre critique du scénario proposé par l'IA (exercice 3) ;
3. la grille de priorisation complétée et vos trois cas d'usage retenus, justifiés (exercice 4) ;
4. la cible v0 (exercice 5).

**Critères :** les chiffres sont exacts, chaque choix est justifié, le niveau de risque AI Act est argumenté.

La cartographie `cartographie_S2.json` sera **reprise automatiquement en séance 3** (gardez les dossiers des séances côte à côte). Si elle manque, un point de reprise est fourni.